<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-02/AI_Modul_2.4_Praktikum_Variabel_dan_Tipe_Data.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 2.4: Praktikum Variabel dan Tipe Data
### Eksplorasi Model Memori Objek, Analisis Presisi Pecahan IEEE 754, Profiling Mutabilitas, dan Parsing Data Telemetri Perkebunan

---

> **Diktat Terkait:** [AI_Modul_2.4_Variabel_dan_Tipe_Data.md](../../docs/part-02/AI_Modul_2.4_Variabel_dan_Tipe_Data.md)  
> **Outputs:** Script analisis alamat memori objek (*memory profiling*), alat uji galat presisi pecahan desimal standar IEEE 754, serta modul *sanitizer* data telemetri panen sawit berbasis konversi tipe defensif.  
> **Outcomes:** Mahasiswa terampil mengaudit tipe data dan mutabilitas objek di memori Heap, menghindari bug *aliasing*, menangani galat pembulatan numerik menggunakan toleransi presisi mesin, serta menulis kode beranotasi *Type Hinting*.  
> **Impacts:** Terciptanya pipeline data AI yang bersih (*data-centric AI*), eliminasi kerugian finansial akibat distorsi angka pecahan timbangan komoditas industri, dan stabilitas operasional pada perangkat komputasi tepi (*Edge IoT*).
>
> **Tujuan Praktikum:**  
> 1. Menginspeksi identitas memori (`id()`), tipe data (`type()`), dan konsumsi memori fisik (`sys.getsizeof()`) berbagai objek Python.  
> 2. Mengukur deviasi presisi bilangan pecahan IEEE 754 dan membandingkan solusi `math.isclose()` versus pustaka `decimal.Decimal`.  
> 3. Membuktikan perbedaan perilaku mutasi *in-place* pada objek mutabel (`list`) dibandingkan objek imutabel (`int`, `float`, `str`).  
> 4. Membangun modul *defensive type casting* untuk membersihkan data mentah heterogen dari stasiun cuaca dan jembatan timbang PKS.

## 1. Persiapan Environment & Pemuatan Library
Jalankan sel berikut untuk memuat modul inspeksi sistem, pustaka matematika desimal presisi tinggi, dan konfigurasi lingkungan:

In [ ]:
# Mengimpor modul inspeksi sistem dan memori
import sys
import math
import decimal

# Mengimpor modul pencatat waktu presisi
import time

# Mengimpor modul anotasi pengetikan terstruktur
from typing import Dict, List, Tuple, Any, Optional

# Mengimpor pustaka saintifik dan visualisasi
import numpy as np
import matplotlib.pyplot as plt

# Mengatur konfigurasi visualisasi grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

# Menampilkan informasi versi Python
print(f"Python Version  : {sys.version.split()[0]}")
print(f"Pointer Size    : {sys.maxsize.bit_length() + 1}-bit Architecture")
print("[OK] Seluruh modul inspeksi data dan memori siap digunakan!")

## 2. Inspeksi Alokasi Memori Objek: Identitas (`id`), Tipe, dan Ukuran Byte
Kita akan membuktikan secara langsung bahwa variabel dalam Python adalah label referensi (*pointer*) yang menunjuk ke objek nyata di memori Heap.

In [ ]:
# Membuat beberapa objek data komoditas perkebunan
tandan_sawit_id = 4500
berat_cpo_ton   = 34.785
kode_sertifikat = "RSPO-2026-INDONESIA"
status_aktif    = True
batch_sensor    = [28.5, 31.0, 29.8]

daftar_variabel = [
    ("tandan_sawit_id", tandan_sawit_id),
    ("berat_cpo_ton", berat_cpo_ton),
    ("kode_sertifikat", kode_sertifikat),
    ("status_aktif", status_aktif),
    ("batch_sensor", batch_sensor)
]

print("=" * 85)
print("PROFIL ALOKASI MEMORI OBJEK PYTHON DI HEAP")
print("=" * 85)
header = f"{'Nama Variabel':<18} | {'Tipe Data':<10} | {'Ukuran (Bytes)':<14} | {'Alamat Memori (Hex)':<18} | {'Nilai'}"
print(header)
print("-" * 85)
for nama, val in daftar_variabel:
    ukuran = sys.getsizeof(val)
    alamat_hex = hex(id(val))
    print(f"{nama:<18} | {type(val).__name__:<10} | {ukuran:<14} | {alamat_hex:<18} | {str(val)}")
print("=" * 85)
print("[INFO] Perhatikan bahwa setiap tipe data memiliki overhead memori metadata bawaan.\n")

## 3. Investigasi Galat Presisi Floating-Point Standar IEEE 754
Sel berikut membuktikan limitasi representasi biner terhadap angka desimal $0.1$ dan membandingkan metode penanganan numerik yang aman.

In [ ]:
# Uji coba klasik penjumlahan pecahan berulang
penjumlahan_desimal = 0.1 + 0.2

print("=" * 75)
print("ANALISIS LIMITASI PECAHAN IEEE 754 (0.1 + 0.2)")
print("=" * 75)
print(f"Nilai Evaluasi (0.1 + 0.2)    : {penjumlahan_desimal:.25f}")
print(f"Nilai Ekspektasi (0.3)        : {0.3:.25f}")
print(f"Komparasi Ketat (0.1 + 0.2 == 0.3) : {penjumlahan_desimal == 0.3} (BAHAYA BUG LOGIKA!)")
print(f"Komparasi Aman (math.isclose)      : {math.isclose(penjumlahan_desimal, 0.3)} (LULUS AMAN)")

# Solusi Finansial & Akuntansi Menggunakan decimal.Decimal
d1 = decimal.Decimal('0.1')
d2 = decimal.Decimal('0.2')
d_sum = d1 + d2
print("\n--- Menggunakan Modul decimal.Decimal (Presisi Absolut) ---")
print(f"Decimal('0.1') + Decimal('0.2') == Decimal('0.3') : {d_sum == decimal.Decimal('0.3')}")
print(f"Hasil Decimal: {d_sum}")
print("=" * 75 + "\n")

## 4. Eksperimen Mutabilitas: Immutability vs In-Place Mutation
Kita membandingkan bagaimana objek mutabel (`list`) dapat dimodifikasi tanpa mengubah alamat memorinya, sedangkan objek imutabel (`int`) selalu mengalokasikan objek baru.

In [ ]:
print("=" * 75)
print("UJI MUTABILITAS 1: OBJEK IMUTABEL (INTEGER)")
print("=" * 75)
skor = 100
id_awal_skor = id(skor)
print(f"Nilai awal : {skor:<5} | Alamat Memori (id): {id_awal_skor}")

# Memodifikasi nilai skor
skor = skor + 1
id_baru_skor = id(skor)
print(f"Nilai baru : {skor:<5} | Alamat Memori (id): {id_baru_skor}")
print(f"Alamat memori berubah? : {id_awal_skor != id_baru_skor} (Objek baru dialokasikan!)")

print("\n" + "=" * 75)
print("UJI MUTABILITAS 2: OBJEK MUTABEL (LIST)")
print("=" * 75)
daftar_truk = ["Truk-A", "Truk-B"]
id_awal_list = id(daftar_truk)
print(f"List awal : {daftar_truk} | Alamat Memori (id): {id_awal_list}")

# Memodifikasi list in-place (append)
daftar_truk.append("Truk-C")
id_baru_list = id(daftar_truk)
print(f"List baru : {daftar_truk} | Alamat Memori (id): {id_baru_list}")
print(f"Alamat memori tetap sama? : {id_awal_list == id_baru_list} (Dimodifikasi in-place!)")
print("=" * 75 + "\n")

## 5. Studi Kasus Agro-Industri: Parser Data Telemetri Penerimaan TBS PKS
Kita mengimplementasikan fungsi pembersih data (*data sanitizer*) yang memproses masukan mentah dari sensor timbangan dan gerbang RFID pabrik kelapa sawit.

In [ ]:
def bersihkan_dan_validasi_transaksi(payload: Dict[str, Any]) -> Tuple[bool, Dict[str, Any], str]:
    """
    Melakukan parsing defensif terhadap data telemetri penimbangan kelapa sawit.
    """
    try:
        # 1. Validasi String Identitas
        trx_id = str(payload.get("trx_id", "")).strip().upper()
        blok   = str(payload.get("blok", "")).strip().upper()
        if not trx_id or not blok:
            return False, {}, "ID Transaksi atau Blok kosong"
        
        # 2. Parsing Defensif Janjang (Integer)
        janjang_val = int(float(payload.get("janjang", 0)))
        if janjang_val <= 0:
            return False, {}, f"Jumlah janjang tidak valid: {janjang_val}"
            
        # 3. Parsing Defensif Berat Kotor dan Tara (Float)
        berat_kotor = float(payload.get("kotor_kg", 0.0))
        berat_tara  = float(payload.get("tara_kg", 0.0))
        if berat_kotor <= berat_tara:
            return False, {}, f"Berat kotor ({berat_kotor}) <= Tara ({berat_tara})"
            
        # 4. Parsing Status Sertifikasi (Boolean)
        raw_cert = payload.get("ispo_certified", False)
        if isinstance(raw_cert, str):
            is_certified = raw_cert.strip().lower() in ("true", "1", "yes", "certified")
        else:
            is_certified = bool(raw_cert)
            
        # Menghitung Berat Bersih dan Berat Janjang Rata-rata (BJR)
        berat_netto = berat_kotor - berat_tara
        bjr = berat_netto / janjang_val
        
        clean_record = {
            "trx_id": trx_id,
            "blok": blok,
            "janjang": janjang_val,
            "netto_kg": round(berat_netto, 2),
            "bjr_kg": round(bjr, 2),
            "ispo": is_certified
        }
        return True, clean_record, "VALID"
        
    except (ValueError, TypeError) as err:
        return False, {}, f"Galat Konversi Tipe Data: {err}"

# Dataset Uji Coba Mentah Heterogen
stream_data_uji = [
    {"trx_id": "trx-101", "blok": "blok-a", "janjang": "140", "kotor_kg": "12500.5", "tara_kg": "4800.0", "ispo_certified": "yes"},
    {"trx_id": "trx-102", "blok": "blok-b", "janjang": 180.0, "kotor_kg": 16200.0, "tara_kg": "4900.5", "ispo_certified": 1},
    {"trx_id": "trx-103", "blok": "blok-c", "janjang": "0",   "kotor_kg": "8000.0",  "tara_kg": "4000.0", "ispo_certified": False},
    {"trx_id": "trx-104", "blok": "blok-d", "janjang": "90",  "kotor_kg": "4000.0",  "tara_kg": "5000.0", "ispo_certified": True},
    {"trx_id": "trx-105", "blok": "blok-e", "janjang": "ABC", "kotor_kg": "10000.0", "tara_kg": "4000.0", "ispo_certified": True}
]

print("=" * 95)
print("LOG PEMROSESAN DEFENSIVE TYPE CASTING PENERIMAAN TBS PKS")
print("=" * 95)
for item in stream_data_uji:
    sukses, rec, status = bersihkan_dan_validasi_transaksi(item)
    if sukses:
        print(f"[BERHASIL] {rec['trx_id']:<10} | Blok: {rec['blok']:<7} | Netto: {rec['netto_kg']:8.1f} kg | "
              f"BJR: {rec['bjr_kg']:5.2f} kg/janjang | ISPO: {'YA' if rec['ispo'] else 'TIDAK'}")
    else:
        t_id = item.get('trx_id', 'UNKNOWN').upper()
        print(f"[DITOLAK]  {t_id:<10} | Alasan: {status}")
print("=" * 95 + "\n")

## 6. Tantangan Pemrograman Mandiri (Scaffolded Challenges)

### Tantangan: Sanitasi Input Berat Janjang TBS (`bersihkan_berat_tbs`)
Buatlah sebuah fungsi `bersihkan_berat_tbs` yang menerima input teks dari operator (yang seringkali mengandung spasi berlebih atau satuan teks seperti `" 18.5 kg "`, `"22.0 KG"`, `" 15 "`) dan mengonversinya menjadi angka `float` murni.

In [ ]:
def bersihkan_berat_tbs(teks_input: str) -> float:
    """
    Membersihkan teks input berat dan mengembalikannya sebagai float murni.
    Melempar ValueError jika masukan tidak valid.
    """
    if not isinstance(teks_input, str):
        raise ValueError("Masukan harus berupa string teks")
        
    # Menghapus spasi dan mengubah ke huruf kecil
    bersih = teks_input.strip().lower()
    
    # Menghapus akhiran satuan 'kg'
    if bersih.endswith("kg"):
        bersih = bersih[:-2].strip()
        
    try:
        nilai_float = float(bersih)
        if nilai_float <= 0.0:
            raise ValueError(f"Berat harus bernilai positif ({nilai_float} <= 0)")
        return nilai_float
    except ValueError as e:
        raise ValueError(f"Format berat tidak dapat dikonversi ke numerik: '{teks_input}'") from e


# Pengujian Unit Mandiri (Unit Tests)
kasus_valid = [
    (" 18.5 kg ", 18.5),
    ("22.0 KG", 22.0),
    (" 15 ", 15.0),
    ("14.75kg", 14.75)
]

print("=" * 70)
print("PENGUJIAN FUNGSI BERSIHKAN_BERAT_TBS")
print("=" * 70)
semua_lulus = True
for raw_str, expected in kasus_valid:
    hasil = bersihkan_berat_tbs(raw_str)
    lulus = math.isclose(hasil, expected)
    print(f"Input: '{raw_str:<10}' -> Output: {hasil:5.2f} kg | [{'LULUS' if lulus else 'GAGAL'}]")
    if not lulus:
        semua_lulus = False

# Uji Kasus Invalid (Harus Melempar ValueError)
kasus_invalid = ["RUSAK", "-5.0 kg", ""]
for raw_bad in kasus_invalid:
    try:
        bersihkan_berat_tbs(raw_bad)
        print(f"Input Bad: '{raw_bad}' -> [GAGAL: Seharusnya melempar ValueError]")
        semua_lulus = False
    except ValueError:
        print(f"Input Bad: '{raw_bad:<10}' -> [LULUS: ValueError berhasil ditangkap]")

print("=" * 70)
if semua_lulus:
    print("[SUCCESS] 100% Kasus Uji Sanitasi Berat TBS Lulus Sempurna!")
else:
    print("[ALERT] Ditemukan kegagalan pada pengujian unit.")

## 7. Rangkuman & Refleksi Praktikum

Dari praktikum ini, kita telah membuktikan bahwa:
1. Variabel dalam Python bekerja sebagai **label referensi pointer** di memori Heap. Pemahaman identitas objek (`id()`) dan mutabilitas krusial untuk mencegah efek samping yang tidak disengaja (*side effects*).
2. Bilangan pecahan desimal berbasis IEEE 754 memiliki batas representasi 52-bit fraksi, sehingga komparasi mutlak `==` pada angka `float` merupakan sumber galat tersembunyi. Penggunaan `math.isclose()` atau `decimal.Decimal` adalah standar wajib industri.
3. Pembersihan tipe data defensif (*defensive casting*) melindungi model AI perkebunan dari data anomali yang dapat menghentikan operasi secara tiba-tiba.

Lanjutkan eksplorasi ke modul berikutnya: **AI Modul 2.5: Operator Matematika dan Logika** untuk mendalami operasi aritmetika lanjut dan vektorisasi data!